In [ ]:
import os
from dotenv import load_dotenv

from typing import List

from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, BaseMessage
from langchain_ollama import ChatOllama, OllamaEmbeddings

load_dotenv(override=True)

BASE_URL:str = os.environ.get("OLLAMA_BASE_URL", "")
EMBEDDING_MODEL: str = "qwen3-embedding:0.6b"
CHAT_MODEL:str = "gemma3:1b"

In [ ]:
persistent_directory = "../db/chroma_db"

# Loading embedding model
embedding = OllamaEmbeddings(base_url=BASE_URL, model=EMBEDDING_MODEL)

# Loading vector database
db = Chroma(
    persist_directory=persistent_directory,
    embedding_function=embedding,
)

In [ ]:
model = ChatOllama(base_url=BASE_URL, model=CHAT_MODEL)

CHAT_HISTORY: List[BaseMessage] = []

In [ ]:
def ask_question(user_query: str) -> str:
    print(f"---- You asked: {user_query} ----")

    if len(CHAT_HISTORY) == 0:
        search_question = user_query
    else:
        messages: List[BaseMessage] = [
            SystemMessage(
                content=f"""Given the chat history, rewrite the new question to be standalone and searchable. Just return the rewritten question. 
                Chat History:
                {CHAT_HISTORY}
                """
            ),
            HumanMessage(content=f"New Question: {user_query}"),
        ]
        result = model.invoke(messages)
        search_question = result.content.strip()  # type: ignore
        print(f"Searching for: {search_question}")

    retriever = db.as_retriever(search_kwargs={"k": 3})
    docs = retriever.invoke(search_question)  # type: ignore

    combined_input = f"""Based on the following documents, please answer this question: {user_query}
    Documents: 
    {"\n".join([f"- {doc.page_content}" for doc in docs])}

    Please provide a clear, helpful answer using only the information from these documents. 
    If you can't find the answer in the documents, say 'I don't have enough information to answer this question.'
    """

    messages = (
        [
            SystemMessage(
                content="You are a helpful assistant that answers questions based on provided documents and conversation history."
            )
        ]
        + CHAT_HISTORY
        + [HumanMessage(content=combined_input)]
    )

    result = model.invoke(messages)
    answer: str = result.content  # type: ignore

    CHAT_HISTORY.append(HumanMessage(content=user_query))
    CHAT_HISTORY.append(AIMessage(content=answer))

    print(f"Answer: {answer}")
    return answer

In [ ]:
CHAT_HISTORY.clear()
ask_question(user_query="Tell me about NVIDIA's latest GPU architecture?")
ask_question(user_query="What's their revenue from it?")